# **1. Object Detection**
Object Detection(객체 탐지)은 이미지나 영상에서 특정 객체의 존재 여부를 확인하고, 해당 객체의 위치를 바운딩 박스(bounding box)로 표시하는 기술입니다.  
이는 컴퓨터 비전에서 중요한 분야로, 이미지 내에서 여러 개의 객체를 동시에 탐지하고 분류할 수 있습니다.  
Object Detection은 주로 딥러닝 기반의 CNN(합성곱 신경망) 모델을 활용하며,   
대표적인 알고리즘으로는 R-CNN 계열(Faster R-CNN, Mask R-CNN), YOLO(You Only Look Once), SSD(Single Shot MultiBox Detector) 등이 있습니다.   
이러한 기술은 자율 주행, 보안 감시, 의료 영상 분석, 증강 현실 등 다양한 분야에서 활용됩니다.

![Object Detection 논문 추천 리스트](./images/논문추천리스트.png)

### 1. 객체 탐지의 출력

- 클래스: 사람, 자동차, 개, 고양이 등
- 바운딩 박스: 객체의 위치와 크기
- confidence score: 해당 예측을 모델이 얼마나 확신하는지 나타내는 점수

> 객체 탐지는 이미지 전체에 하나의 클래스만 붙이는 Image Classification과 다름. 한 이미지 안에 여러 객체가 있어도 각각 따로 찾아내고 점수를 부여할 수 있음.

### 2. 대표적인 딥러닝 기반 계열

1. One-Stage
    - 한 번의 네트워크 흐름에서 객체 클래스와 위치를 직접 예측
    - 일반적으로 속도가 빠르며, 실시간 응용에 많이 사용
    - YOLO, SSD, RetinaNet 등
2. Two-Stage
    - 먼저 객체 후보 영역을 만들고, 그 후보를 다시 분류하고 박스를 보정함
    - 일반적으로 정밀한 탐지에 강점이 있음
    - Faster R-CNN, Mask R-CNN 등

> 실무에서는 자율주행, CCTV 분석, 제조 불량 검출, 의료영상 분석 등에서 사용. 현대 모델에서는 "One-Stage"는 조금 부정확하고 "Two-Stage"는 정확하다라는 단정을 하면 안됨.  
> 데이터셋, 모델 크기, 학습 방식에 따라 성능 관계는 달라짐

# **2. YOLO**
YOLO(You Only Look Once)는 이미지를 격자 단위로 나누어 각 격자에서 객체의 위치(바운딩 박스 좌표)와 클래스를 동시에 예측하는 1단계 객체 탐지(One-Stage Detection) 알고리즘입니다.  
한 번의 신경망 추론으로 전체 이미지의 탐지를 수행하기 때문에 매우 빠르고, 실시간 객체 탐지에 적합하다는 장점이 있습니다.  
YOLO 계열은 v1에서 시작해 v3까지는 Joseph Redmon이 개발했으며, 이후 v4·v7은 커뮤니티 연구자들, v5·v8·v11은 Ultralytics가 주도적으로 발전시켜 현재까지 이어지고 있습니다.  
이러한 발전 과정을 거치면서 YOLO는 속도와 정확도의 균형을 잡은 대표적인 객체 탐지 모델로 자리 잡아 다양한 산업 현장에서 활용되고 있습니다.

### 1. YOLO 모델 크기별 분류
- n (nano): 초경량 모델로 모바일 및 임베디드 장치에서 사용하기 적합합니다.
- s (small): 속도가 빠르면서도 정확도가 준수하여 실시간 추론이 필요한 경우 사용합니다.
- m (medium): 균형 잡힌 모델로 중간 정도의 성능과 속도를 제공합니다.
- l (large): 높은 정확도를 요구하는 애플리케이션에서 사용합니다.
- x (extra-large): 최대 성능을 발휘하지만, 속도가 상대적으로 느립니다.

### 2. YOLO 모델 유형별 분류
- YOLO Detect: 가장 일반적인 객체 탐지 모델로, 특정 객체의 위치와 크기를 바운딩 박스로 반환합니다.
- YOLO Segment: 객체 탐지뿐만 아니라 픽셀 단위의 분할(Segmentation)까지 수행하는 모델입니다.
- YOLO Pose: 사람의 관절 위치를 탐지하여 포즈를 예측하는 모델입니다.
- YOLO Cls: 이미지를 분류하는 모델입니다.
- YOLO OBB: 이미지를 회전하여 객체를 탐지하는 모델입니다.

# **3. PascalVOC 2007**
PascalVOC 2007은 객체 탐지(Object Detection), 분할(Segmentation), 동작 인식(Action Recognition) 등의 다양한 컴퓨터 비전 과제를 위한 벤치마크 데이터셋입니다.  
총 20개의 객체 클래스(예: 사람, 자동차, 개, 고양이 등)를 포함하며, 훈련(train), 검증(val), 테스트(test) 세트로 구성되어 있습니다.  
각 이미지에는 객체의 경계 상자(Bounding Box) 및 해당 클래스 레이블이 주어지며, Mean Average Precision(mAP) 평가 기준을 적용하여 성능을 측정합니다.

In [1]:
import sys
import torch
from pathlib import Path

In [2]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

cpu


In [3]:
PROJECT_ROOT = Path(r'C:\SHJ\KDT\11_CV')
print(PROJECT_ROOT)

C:\SHJ\KDT\11_CV


In [4]:
DATA_DIR = PROJECT_ROOT / "data"
VOC_ROOT = DATA_DIR / "VOCdevkit" / "VOC2007"
YOLO_ROOT = DATA_DIR / 'Custom_YOLO'

ANNOTATIONS_DIR = VOC_ROOT / "Annotations"
IMAGES_DIR = VOC_ROOT / "JPEGImages"

print("DATA DIR:", DATA_DIR.resolve())
print("VOC ROOT:", VOC_ROOT.resolve())
print("YOLO ROOT:", YOLO_ROOT.resolve())

DATA DIR: C:\SHJ\KDT\11_CV\data
VOC ROOT: C:\SHJ\KDT\11_CV\data\VOCdevkit\VOC2007
YOLO ROOT: C:\SHJ\KDT\11_CV\data\Custom_YOLO


In [5]:
for sub in [
    'images/train2007', 'images/val2007', 'images/test2007',
    'labels/train2007', 'labels/val2007', 'labels/test2007'
]:
    (YOLO_ROOT/sub).mkdir(parents=True, exist_ok=True)

# **4. YOLO annotation 기본 구조**

```
<class_id> <x_center> <y_center> <width> <height>
예) 6 0.191130 0.810886 0.380460 0.378168
```
- class_id: 0부터 시작하는 클래스 번호
- x_center, y_center: 박스의 중심점
- width, height: 박스의 너비와 높이
- 좌표는 모두 이미지 크기로 나눈 0~1 사이 값

> VOC XML의 (xmin, ymin, xmax, ymax)를 YOLO 형식으로 바꿀 때는 중심점과 크기를 계산한 뒤 이미지 너비와 높이로 정규화해야 함

In [6]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

CLASS_TO_ID = {name: idx for idx, name in enumerate(VOC_CLASSES)}
CLASS_TO_ID

{'aeroplane': 0,
 'bicycle': 1,
 'bird': 2,
 'boat': 3,
 'bottle': 4,
 'bus': 5,
 'car': 6,
 'cat': 7,
 'chair': 8,
 'cow': 9,
 'diningtable': 10,
 'dog': 11,
 'horse': 12,
 'motorbike': 13,
 'person': 14,
 'pottedplant': 15,
 'sheep': 16,
 'sofa': 17,
 'train': 18,
 'tvmonitor': 19}

In [7]:
import xml.etree.ElementTree as ET

In [8]:
def voc_box_to_yolo(xmin, ymin, xmax, ymax, image_w, image_h):
    x_center = ((xmin + xmax) / 2.0) / image_w
    y_center = ((ymin + ymax) / 2.0) / image_h
    box_w = (xmax - xmin) / image_w
    box_h = (ymax - ymin) / image_h
    return x_center, y_center, box_w, box_h

In [9]:
def convert_voc_xml(xml_path, label_path):
    root = ET.parse(xml_path).getroot()
    size = root.find('size')
    image_w = float(size.findtext('width'))
    image_h = float(size.findtext('height'))

    lines = []
    for obj in root.findall('object'):
        class_name = obj.findtext('name')
        if class_name not in CLASS_TO_ID:
            continue
        bnd = obj.find('bndbox')
        xmin = float(bnd.findtext('xmin'))
        ymin = float(bnd.findtext('ymin'))
        xmax = float(bnd.findtext('xmax'))
        ymax = float(bnd.findtext('ymax'))

        xmin = max(0.0, min(xmin, image_w))
        xmax = max(0.0, min(xmax, image_w))
        ymin = max(0.0, min(ymin, image_h))
        ymax = max(0.0, min(ymax, image_h))

        if xmax <= xmin or ymax <= ymin:
            continue

        x, y, w, h = voc_box_to_yolo(xmin, ymin, xmax, ymax, image_w, image_h)
        class_id = CLASS_TO_ID[class_name]
        lines.append(f'{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}')

    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text('\n'.join(lines), encoding='utf-8')

In [10]:
xml_path = Path(r'C:\SHJ\KDT\11_CV\data\VOCdevkit\VOC2007\Annotations\000001.xml')
label_path = Path(r'C:\SHJ\KDT\11_CV\data\000001.txt')

convert_voc_xml(xml_path, label_path)

In [11]:
import shutil

def prepare_split(voc_dir, split_name, output_name):
    split_file = voc_dir / 'imageSets' / 'Main' / f'{split_name}.txt'
    image_dir = voc_dir / 'JPEGImages'
    annotation_dir = voc_dir / 'Annotations'
    out_image_dir = YOLO_ROOT / 'images' / output_name
    out_label_dir = YOLO_ROOT / 'labels' / output_name
    out_image_dir.mkdir(parents=True, exist_ok = True)
    out_label_dir.mkdir(parents=True, exist_ok = True)

    image_ids = [x.strip() for x in split_file.read_text(encoding='utf-8').splitlines() if x.strip()]
    # print(image_ids)
    for image_id in image_ids:
        src_image = image_dir / f'{image_id}.jpg'
        src_xml = annotation_dir / f'{image_id}.xml'
        shutil.copy2(src_image, out_image_dir / src_image.name)
        convert_voc_xml(src_xml, out_label_dir / f'{image_id}.txt')
    print(f'{output_name}: {len(image_ids):,} images prepared')


In [12]:
prepare_split(VOC_ROOT, 'train', 'train2007')

train2007: 2,501 images prepared


In [13]:
prepare_split(VOC_ROOT, 'val', 'val2007')

val2007: 2,510 images prepared


In [15]:
prepare_split(VOC_ROOT, 'test', 'test2007')

test2007: 4,952 images prepared


# **5. Dataset YAML**
Dataset YAML 파일은 데이터셋의 위치와 클래스 이름을 모델에게 알려주는 설정 파일
- Dataset YAML: train/val/test 경로, 클래스 이름
- Model YAML: 모델 architecture 정의

> 학습할 떄 항상 YAML 파일을 만들어야 하는 것은 아님. 일반적인 transfer learning에서는 pretrained.pth 가중치를 불러오고 dataset YAML만 지정하면 편하게 사용할 수 있음

```
pip install ultralytics
pip install pyyaml
```

In [16]:
import yaml
from ultralytics import YOLO

In [17]:
custom_voc_yaml = YOLO_ROOT / 'custom_voc.yaml'

data_config = {
    "path": str(YOLO_ROOT.resolve()),
    'train': 'images/train2007',
    'val': 'images/val2007',
    'test': 'images/test2007',
    'names': {i: name for i, name in enumerate(VOC_CLASSES)}
}

with open(custom_voc_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_config, f, allow_unicode=True, sort_keys=False)

print('생성된 YAML 경로:', custom_voc_yaml)
print(custom_voc_yaml.read_text(encoding='utf-8'))

생성된 YAML 경로: C:\SHJ\KDT\11_CV\data\Custom_YOLO\custom_voc.yaml
path: C:\SHJ\KDT\11_CV\data\Custom_YOLO
train: images/train2007
val: images/val2007
test: images/test2007
names:
  0: aeroplane
  1: bicycle
  2: bird
  3: boat
  4: bottle
  5: bus
  6: car
  7: cat
  8: chair
  9: cow
  10: diningtable
  11: dog
  12: horse
  13: motorbike
  14: person
  15: pottedplant
  16: sheep
  17: sofa
  18: train
  19: tvmonitor



# **6.Pretrained 모델 선택**

In [18]:
model = YOLO('yolo11s.pt')
model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentu

In [19]:
results = model.train(
    data = str(custom_voc_yaml),
    epochs = 10,
    imgsz = 640,
    batch = 16,
    device = DEVICE,
    workers = 2,
    # project = str(PROJECT_ROOT / 'runs' / 'detect'),
    name = 'voc2007_yolo11s',
    exist_ok = True,
    seed = 2026
)

Ultralytics 8.4.143  Python-3.12.10 torch-2.14.0+cpu CPU (12th Gen Intel Core i7-12700)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\SHJ\KDT\11_CV\data\Custom_YOLO\custom_voc.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=voc2007_yolo11s, nbs=

# **7. 객체 탐지 성능 평가 지표**
- 객체 탐지는 단순한 accuracy 만으로 성능을 평가하기 어렵다.
- 분류 문제에서는 보통 클래스가 맞았는지 확인하면 되지만, 객체 탐지에서는 두 가지를 동시에 확인해야 함.
    1. 예측 클래스가 정답 클래스와 같은지 비교
    2. 예측한 Bounding Box가 실제 객체 위치와 충분히 겹치는지 확인

### 1. IoU(Intersection over Union)
IoU는 객체 탐지에서 예측 Bounding Box와 정답 Bounding Box의 위치가 얼마나 비슷한지 측정하는 지표
- Intersection(교집합): 예측 박스와 정답 박스가 실제로 겹치는 영역
- Union(합집합): 두 박스를 합쳤을 때 차지하는 전체 영역
- IoU: 교집합 면적 / 합집합 면적

> 객체 탐지에서 클래스가 맞아도 Bounding Box가 객체와 거의 겹치지 않는다면 올바른 탐지라고 보기 어려움. 따라서 평가할 때 일정 IoU이상을 만족한 예측만 TP로 인정함.

```
IoU = 1            # 예측 박스와 정답 박스가 완전히 일치
IoU가 1에 가까움    # 위치를 매우 정확하게 예측
IoU가 작음          # 두 박스가 조금만 겹치거나 위치가 많이 어긋남
```

> AP50의 50은 대표적으로 IoU 0.50을 기준으로 TP/FP를 판단하여 AP를 계산한다는 의미

### 2. Confidence Score
- 모델이 이 예측에 대체 얼마나 믿고 있는가를 나타내는 점수
- 예) person 0.92 > 이 곳에 사람이 있다고 판단했고, 이 예측에 대한 신뢰도가 높다
- YOLO는 이미지에서 딱 하나의 박스만 만드는 것이 아님. 객체일 가능성이 있는 많은 후보 박스를 예측
```
후보1 > person 0.94
후보2 > person 0.85
후보3 > person 0.64
후보4 > dog 0.08
```
- confidence threshold(신뢰도 임계값)을 사용하여 수 많은 예측 중에서 믿을 만한 예측을 걸러냄

> Confidence Score가 높을수록 모델이 해당 탐지를 더 강하게 믿고 있다는 의미

### AP(Average Precision)
- 한 클래스에 대해 모델의 Precision과 Recall 성능을 하나의 값으로 요약한 지표
- 객체 탐지에서는 각 예측마다 confidence score가 있음. 평가기는 높은 confidence 예측부터 낮은 confidence 예측까지 순서대로 보면서 Precision과 Recall의 변화를 계산 > PR Curve를 만들 수 있음
    - 가로축: Recall
    - 세로축: Precision
- AP 계산
    1. 해당 클래스의 모든 예측 결과를 모음
    2. confidence가 높은 순서로 정렬
    3. 각 예측을 정답 박스와 비교
    4. 클래스와 IoU 조건을 이용해 TP 또는 FP 결정
    5. 예측을 하나씩 포함시키면서 Precision과 Recall 계산
    6. Precision-Recall Curve 구성
    7. 전체를 요약하여 AP 계산
- AP50: IoU 기준을 0.50으로 두었을 때 계산한 AP
- aP75: IoU 기준을 0.75으로 두었을 때 계산한 AP

> IoU 기준이 높아질수록 Bounding Box의 위치가 더 정확해야 TP로 인정됨. 따라서 AP75가 AP50보다 더 엄격한 평가

### mAP(mean Average Precision)
객체 탐지 데이터셋에는 여러 클래스가 있으므로, 클래스 AP를 구한 뒤 평균을 내어 전체 모델 성능을 표현한 값

- mAP50: IoU=0.50 기준에서 각 클래스의 AP를 구한 뒤 클래스 전체 평균을 낸 값. Bounding Box가 정답과 어느 정도만 겹쳐도 TP로 인정되기 때문에 상대적으로 점수가 높게 나올 수 있음
- mAP50-95: IoU 기준을 0.50, 0.55, 0.60, ..., 0.90, 0.95 각 IoU 기준에서 AP를 계산하고, 클래스와 IoU 기준 전체에 대해 평균을 냄

> mAP50-95는 단순히 객체를 대략 찾는 것 뿐 아니라 Bounding Box의 위치를 얼마나 정밀하게 맞추는지까지 더 엄격하게 반영

- 평가
    - mAP50은 높은데 mAP50-95가 상대적으로 많이 낮음 > 객체 자체는 잘 찾지만 Bounding Box 위치 정밀도가 부족할 가능성
    - 둘 다 낮음 > 분류, 위치 예측, 데이터 품질, 학습량 등 전체적인 점검 필요

> 전체 mAP만 보지 말고 클래스별 AP도 꼭 함께 확인. 특정 클래스만 유독 성능이 낮을 수 있음

### NMS
YOLO의 predict(..., iou=...) 또는 검증 설정에서 등장하는 iou는 겹쳐 있는 예측 박스를 정리하는 NMS용 threshold
- 평가 IoU: 예측 박스와 정답 박스를 비교
- NMS IoU: 여러 예측 박스끼리 비교하여 중복 탐지를 제거

### FPS와 Latency
- FPS (Frames Per Second): 1초에 처리 가능한 프레임 수
- Latency: 한 입력이 결과로 나오기까지 걸리는 시간

### 실무 성능 측정 시 고려해야할 사항
- GPU / CPU / NPU 종류
- batch size
- image size
- FP32 / FP16 / INT8 / INT1 정밀도
- preprocessiong 포함 여부

> 객체 탐지 모델은 정확도만 높다고 좋은 모델이 아닐 수 있음. 실제 서비스에서는 정확도 + 속도 + 메모리 사용량 + 배포 장비를 함께 고려해야 함

# **8. 학습 중 Epoch에 나오는 항목**

### 1. Epoch
전체 학습 데이터를 한 번 모두 학습한 것

### 2. GPU_mem
현재 학습 과정에서 사용 중인 GPU 메모리(VRAM)와 관련된 값

### 3. box_loss
정답 박스가 있는데 모델이 다른 위치에 박스를 그렸다면 손실이 발생함 (모델이 객체의 위치를 얼마나 잘 찾고 있는지에 대한 손실 값)

### 4. cls_loss
객체 종류를 얼마나 잘 맞히는지와 관련된 손실

### 5. dfl_loss
- dfl (Distribution focal loss)
- Bounding Box의 경계 위치를 보다 정밀하게 예측하기 위한 Loss
    - box_loss: 박스 전체 위치가 정답과 얼마나 잘 맞는지
    - dfl_loss: 박스의 경계를 얼마나 정밀하게 잡는지

### 6. Instances
- 이미지 개수가 아니라 객체의 개수
- 한 이미지 안에도 여러 객체가 있을 수 있음

### 7. Size
모델 학습에 사용하는 입력 이미지의 크기

### 8. Epoch가 끝나면 나오는 Validation 결과
```
Class: 전체 클래스를 종합해서 평가한 결과
Images: 검증에 사용된 이미지 수
Instances: 검증 이미지 안에 존재하는 정답 객체 개수
Precision: YOLO가 있다고 한 것 중에서 진짜가 얼마나 되는지? 오탐이 적을수록 높아짐 (FP와 관계)
Recall: 실제로 존재하는 객체를 YOLO가 얼마나 놓치지 않고 찾았는가? 놓친 객체가 적을수록 높아짐 (FN과 관계)
mAP50: IoU 0.5를 기준으로 평가. 정답 박스와 예측 박스가 어느 정도 겹치는지 계산
mAP50-95: IoU 0.50 ~ 0.95까지 AP를 계산한 뒤 평균을 냄.
```

# **9. runs 폴더**

### 1. weights
- best.pt: 검증 성능이 가장 좋았던 시점의 모델. 실제로 서비스하거나 추론할 때는 best.pt를 사용
- last.pt: 마지막 Epoch가 끝난 시점의 모델. 중단된 학습을 이어서 진행할 때 사용

### 2. args.yaml
- 학습할 때 사용했던 설정값을 저장한 파일
- 이 모델을 어떤 설정으로 학습했는지 확인할 수 있는 기록
- 실무에서 실험 재현을 위해 사용하는 파일

### 3. results.csv
- Epoch마다 나온 학습 결과를 표 형태로 저장한 파일
- 데이터를 직접 분석할 때 사용

### 4. results.png
- 학습 전체 과정을 그래프로 보여줌
- 예) train/box_loss, cls_loss, dfl_lss / val/box_loss, cls_loss, dfl_loss ...

### 5. confusion_matrix.png
- 모델이 어떤 클래스를 어떤 클래스로 착각하는지 확인할 수 있는 혼동 행렬

### 6. labels.jpg
- 학습 데이터의 Label 분포를 시각화한 이미지
- 어떤 클래스의 객체가 많은지, Bounding Box의 위치나 크기가 어떻게 분포되어 있는지 등을 파악하는데 도움을 줌

In [26]:
best_path = Path('./runs/detect/voc2007_yolo11s/weights/best.pt')
model = YOLO(str(best_path))

FileNotFoundError: [Errno 2] No such file or directory: 'runs\\detect\\voc2007_yolo11s\\weights\\best.pt'

In [27]:
val_results = model.val(
    data = str(custom_voc_yaml),
    split = 'val',
    imgsz = 640,
    batch = 16,
    device = DEVICE,
    workers = 2
)

YOLO11s summary (fused): 100 layers, 9,420,540 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1691.1372.3 MB/s, size: 82.3 KB)
val: Scanning C:\SHJ\KDT\11_CV\data\Custom_YOLO\labels\val2007.cache... 2510 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2510/2510  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 34% ━━━━──────── 54/157 1.6s/it 1:03<2:464


KeyboardInterrupt: 

In [ ]:
print('mAP50:', val_results.box.map50)
print('mAO50-95:', val_results.box.map)

In [ ]:
val_results = model.val(
    data = str(custom_voc_yaml),
    split = 'test',
    imgsz = 640,
    batch = 16,
    device = DEVICE,
    workers = 2
)